# 🧩 構成解決とフラグ実在

実 install 後に ModelConfig / SpeculativeConfig / argparse を通す。


In [ ]:
import subprocess
import sys

print("=== 素の環境 ===")
r = subprocess.run([sys.executable, "-c",
                    "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("  ", (r.stdout or r.stderr).strip()[:200])

print("=== pip install vllm==0.32.0 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "vllm==0.32.0"], capture_output=True, text=True, timeout=5400)
tail = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
print("   rc:", r.returncode, "|", " / ".join(tail[-3:])[:300])
r = subprocess.run([sys.executable, "-c",
                    "import vllm, torch, transformers; print(vllm.__version__, torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("   versions:", (r.stdout or r.stderr).strip()[:300])

def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

ROOT = find_root()
TPL = ROOT + "/chat_template.jinja"
print("ROOT:", ROOT)
print("template exists:", __import__("os").path.isfile(TPL))

CASE = r"""
import json, sys, traceback
ROOT, TPL = sys.argv[1], sys.argv[2]
from vllm.config import ModelConfig, SpeculativeConfig


def show(tag, fn):
    try:
        v = fn()
        print(f"[OK] {tag}")
        return v
    except Exception as e:
        print(f"[NG] {tag}: {type(e).__name__}: {str(e)[:600]}")
        return None


def dumpmc(tag, mc):
    hf = getattr(mc, "hf_config", None)
    qc = getattr(hf, "quantization_config", None)
    qc = qc if isinstance(qc, dict) else {}
    mm = getattr(mc, "multimodal_config", None)
    print(f"    quant={getattr(mc, 'quantization', None)!r} "
          f"model_type={getattr(hf, 'model_type', None)!r} dtype={mc.dtype}")
    print(f"    arch={getattr(hf, 'architectures', None)}")
    print(f"    qc.method={qc.get('quant_method')!r} qc.format={qc.get('format')!r}")
    print(f"    mm={type(mm).__name__ if mm else None} "
          f"lang_only={getattr(mm, 'language_model_only', None)}")
    print(f"    gen_cfg={getattr(mc, 'generation_config', None)!r}")


print("=== ModelConfig ===")
base = show("base", lambda: ModelConfig(model=ROOT, dtype="half", max_model_len=8192,
                                        trust_remote_code=True))
if base:
    dumpmc("base", base)
    print("    diff_sampling:", base.get_diff_sampling_param())
    print("    hf mtp_num_hidden_layers:",
          getattr(base.hf_config, "mtp_num_hidden_layers", None))
    print("    hf num_hidden_layers:", getattr(base.hf_config, "num_hidden_layers", None))

for q in ("compressed-tensors", "awq", "auto"):
    mc = show(f"quant={q}", lambda q=q: ModelConfig(
        model=ROOT, dtype="half", max_model_len=8192, quantization=q,
        trust_remote_code=True))
    if mc:
        dumpmc(f"quant={q}", mc)

mc = show("language_model_only", lambda: ModelConfig(
    model=ROOT, dtype="half", max_model_len=8192, language_model_only=True,
    trust_remote_code=True))
if mc:
    dumpmc("lang_only", mc)
    mm = getattr(mc, "multimodal_config", None)
    print("    limit_per_prompt:", getattr(mm, "limit_per_prompt", None))

mc = show("generation_config=vllm", lambda: ModelConfig(
    model=ROOT, dtype="half", max_model_len=8192, generation_config="vllm",
    trust_remote_code=True))
if mc:
    print("    diff_sampling(vllm):", mc.get_diff_sampling_param())

mc = show("override_generation_config", lambda: ModelConfig(
    model=ROOT, dtype="half", max_model_len=8192,
    override_generation_config={"repetition_penalty": 1.1},
    trust_remote_code=True))
if mc:
    print("    diff_sampling(override):", mc.get_diff_sampling_param())

print("\n=== SpeculativeConfig（method 別）===")
if base:
    cases = [("mtp", dict(num_speculative_tokens=1)),
             ("mtp", dict(num_speculative_tokens=3)),
             ("mtp", dict(num_speculative_tokens=1, model=ROOT)),
             ("qwen3_5_mtp", dict(num_speculative_tokens=1)),
             ("mtp", dict()),
             ("mtp", dict(num_speculative_tokens=0)),
             ("mtp", dict(num_speculative_tokens=-1))]
    for meth, kw in cases:
        tag = f"{meth} {kw}"
        sp = show(tag, lambda meth=meth, kw=kw: SpeculativeConfig(
            method=meth, target_model_config=base, **kw))
        if not sp:
            continue
        dm = getattr(sp, "draft_model_config", None)
        print(f"    -> n={sp.num_speculative_tokens} model={sp.model!r} "
              f"quant={getattr(sp, 'quantization', None)!r}")
        if dm is not None:
            print(f"    draft: model={dm.model!r} "
                  f"model_type={getattr(dm.hf_config, 'model_type', None)!r} "
                  f"quant={dm.quantization!r}")
            print(f"    draft arch={getattr(dm.hf_config, 'architectures', None)}")
            print(f"    draft mtp_num_hidden_layers="
                  f"{getattr(dm.hf_config, 'mtp_num_hidden_layers', None)}")

print("\n=== chat template ===")
from vllm.renderers.chat_utils import validate_chat_template
show("validate(file)", lambda: validate_chat_template(TPL) or "ok")
show("validate(nonexistent)", lambda: validate_chat_template("/nope.jinja"))

from vllm.transformers_utils.tokenizer import get_tokenizer
tok = show("get_tokenizer", lambda: get_tokenizer(ROOT, trust_remote_code=True))
if tok:
    t = tok.get_chat_template(None, tools=None)
    print("    template len:", len(t) if t else None, "| <think>:", "<think>" in (t or ""))

print("\n=== 実 argparse（フラグ実在の本判定）===")
from vllm.entrypoints.launchers.cli_args import (
    make_arg_parser, validate_parsed_serve_args)
from vllm.utils.argparse_utils import FlexibleArgumentParser

p = make_arg_parser(FlexibleArgumentParser())

SPEC_JSON = json.dumps({"method": "mtp", "num_speculative_tokens": 1})
FLAGS = [
    ("--host", ["127.0.0.1"]),
    ("--port", ["8000"]),
    ("--served-model-name", ["m"]),
    ("--tensor-parallel-size", ["2"]),
    ("--dtype", ["half"]),
    ("--quantization", ["compressed-tensors"]),
    ("--max-model-len", ["8192"]),
    ("--max-num-seqs", ["4"]),
    ("--max-num-batched-tokens", ["4096"]),
    ("--gpu-memory-utilization", ["0.9"]),
    ("--kv-cache-dtype", ["fp8"]),
    ("--swap-space", ["0"]),
    ("--cpu-offload-gb", ["0"]),
    ("--trust-remote-code", []),
    ("--reasoning-parser", ["qwen3"]),
    ("--enforce-eager", []),
    ("--enable-prefix-caching", []),
    ("--enable-auto-tool-choice", []),
    ("--tool-call-parser", ["qwen3_xml"]),
    ("--speculative-config", [SPEC_JSON]),
    ("--spec-method", ["mtp"]),
    ("--language-model-only", []),
    ("--limit-mm-per-prompt", ['{"image":0}']),
    ("--chat-template", [TPL]),
    ("--chat-template-content-format", ["string"]),
    ("--default-chat-template-kwargs", ['{"reasoning_effort":"medium"}']),
    ("--trust-request-chat-template", []),
    ("--disable-log-requests", []),
    ("--enable-log-requests", []),
    ("--enable-chunked-prefill", []),
    ("--compilation-config", ['{"level":0}']),
    ("--load-format", ["safetensors"]),
    ("--mm-processor-kwargs", ["{}"]),
    ("--disable-mm-preprocessor-cache", []),
    ("--uvicorn-log-level", ["warning"]),
    ("--api-key", ["k"]),
    ("--allowed-origins", ['["*"]']),
    ("--tokenizer", [ROOT]),
    ("--seed", ["0"]),
    ("--distributed-executor-backend", ["mp"]),
    ("--worker-cls", [""]),
    ("--disable-custom-all-reduce", []),
    ("--generation-config", ["auto"]),
    ("--override-generation-config", ['{"repetition_penalty":1.05}']),
    ("--enforce-sampling", []),
    ("--served-model-name-extra", ["x"]),
]

ok_flags, ng_flags = [], []
print("結果は「parse できるか」で判定。--help の grep より確実。")
for flag, vals in FLAGS:
    try:
        p.parse_args([ROOT] + [flag] + vals)
        ok_flags.append(flag)
        print(f"  [OK] {flag}")
    except SystemExit as e:
        ng_flags.append(flag)
        print(f"  [NG] {flag} (SystemExit {e.code})")
    except Exception as e:
        ng_flags.append(flag)
        print(f"  [NG] {flag} ({type(e).__name__}: {str(e)[:160]})")

print("\n--- まとめ ---")
print("OK:", len(ok_flags), "NG:", len(ng_flags))
print("OK_FLAGS =", json.dumps(ok_flags, ensure_ascii=False))
print("NG_FLAGS =", json.dumps(ng_flags, ensure_ascii=False))

print("\n=== 全部盛り 1 回の parse ===")
ALL = [ROOT, "--served-model-name", "m", "--host", "127.0.0.1", "--port", "8000",
       "--tensor-parallel-size", "2", "--dtype", "half", "--max-model-len", "8192",
       "--max-num-seqs", "4", "--gpu-memory-utilization", "0.9",
       "--kv-cache-dtype", "fp8", "--trust-remote-code",
       "--reasoning-parser", "qwen3", "--enable-auto-tool-choice",
       "--tool-call-parser", "qwen3_xml", "--speculative-config", SPEC_JSON,
       "--language-model-only", "--enforce-eager", "--chat-template", TPL,
       "--chat-template-content-format", "string",
       "--default-chat-template-kwargs", '{"reasoning_effort":"medium"}',
       "--generation-config", "auto", "--disable-log-requests"]
ns = show("parse(ALL)", lambda: p.parse_args(ALL))
if ns:
    for k in ("chat_template", "chat_template_content_format",
              "default_chat_template_kwargs", "language_model_only",
              "speculative_config", "reasoning_parser", "tool_call_parser",
              "generation_config", "host", "port"):
        print(f"    {k} = {getattr(ns, k, '<missing>')!r}")
    show("validate_parsed_serve_args", lambda: validate_parsed_serve_args(ns) or "ok")
"""

import os
import subprocess
import sys
import tempfile

case = os.path.join(tempfile.gettempdir(), "cfgcase.py")
open(case, "w").write(CASE)
r = subprocess.run([sys.executable, case, ROOT, TPL],
                   capture_output=True, text=True, timeout=3600)
print((r.stdout or "")[-12000:])
if r.returncode != 0:
    print("=== stderr ===")
    print((r.stderr or "")[-4000:])
